In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv

--2026-10-05 20:00:03--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 635180 (620K) [text/plain]
Saving to: ‘car_fuel_efficiency_2026.csv’

car_fuel_efficiency 100%[===================>] 620.29K  --.-KB/s    in 0.03s   

2026-10-05 20:00:03 (20.8 MB/s) - ‘car_fuel_efficiency_2026.csv’ saved [635180/635180]



In [3]:
df = pd.read_csv("/kaggle/working/car_fuel_efficiency_2026.csv")
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [4]:
#Q1
features = ["engine_displacement", "horsepower", "vehicle_weight", "model_year"]
target = ["fuel_efficiency_mpg"]
df = df[features + target]
df.isnull().any()

engine_displacement    False
horsepower              True
vehicle_weight         False
model_year             False
fuel_efficiency_mpg    False
dtype: bool

In [5]:
#Q2
hp_median = df['horsepower'].median()
hp_median

254.0

In [6]:
def split_dataset(df, seed=42):
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test
    
    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)
    
    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]
    df_test = df.iloc[idx[n_train + n_val:]]

    return df_train, df_val, df_test

In [7]:
def linear_regression(X, w_tuple):
    w0, w = w_tuple
    return w0 + X.dot(w)

def train_linear_regression(X, y, r):    
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])
    
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)

    return w[0], w[1:]

def rmse(y, y_pred):
    se = (y - y_pred) ** 2
    mse = se.mean()
    return np.sqrt(mse)

In [8]:
def evaluate(df_train, df_val, fill_value=0, r=0):
    y_train = df_train[target].to_numpy()
    y_val = df_val[target].to_numpy()

    X_train = df_train[features].fillna(fill_value).to_numpy()
    X_val = df_val[features].fillna(fill_value).to_numpy()

    w = train_linear_regression(X_train, y_train, r=r)

    y_pred = linear_regression(X_val, w)
    score = rmse(y_val, y_pred)

    return float(score)

In [9]:
#Q3
df_train, df_val, df_test = split_dataset(df, seed=42)

hp_mean = df_train['horsepower'].mean()

score_mean = evaluate(df_train, df_val, fill_value=hp_mean)
score_zero = evaluate(df_train, df_val, fill_value=0)

round(score_mean, 3), round(score_zero, 3)

(2.202, 2.205)

In [10]:
#Q4
reg_vals = [0, 0.01, 0.1, 1, 5, 10, 100]

min_score = float('inf')
best_r = None
for r in reg_vals:
    score = evaluate(df_train, df_val, fill_value=0, r=r)
    print(f"r = {r} -> rmse = {round(score, 4)}")
    if score < min_score:
        min_score = score
        best_r = r

print(f"Best r = {best_r} with rmse score = {round(min_score, 4)}")

r = 0 -> rmse = 2.2053
r = 0.01 -> rmse = 2.2058
r = 0.1 -> rmse = 2.2241
r = 1 -> rmse = 2.3492
r = 5 -> rmse = 2.4094
r = 10 -> rmse = 2.4195
r = 100 -> rmse = 2.4292
Best r = 0 with rmse score = 2.2053


In [11]:
seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
rmse_scores = []

for seed in seeds:
    df_train, df_val, df_test = split_dataset(df, seed=seed)    
    score = evaluate(df_train, df_val, fill_value=0, r=0)
    rmse_scores.append(score)

std_dev = np.std(rmse_scores)
result = round(std_dev, 3)

print(f"rmse scores: {rmse_scores}")
print(f"std_dev: {result}")

rmse scores: [2.239204143046126, 2.2021128210838845, 2.1634197787530947, 2.2043588768539224, 2.2018800943311554, 2.2457851509085134, 2.2697212079524043, 2.190794599933443, 2.216611201686444, 2.2070365928178957]
std_dev: 0.029


In [12]:
df_train, df_val, df_test = split_dataset(df, seed=9)

df_train = pd.concat([df_train, df_val])
score = evaluate(df_train, df_test, fill_value=0, r=0.001)
round(score, 3)

2.236